# Thinking Twice: the Math of Test-Time Compute
Companion notebook for the 2026-09-27 post. We reproduce — with exact binomial math,
no training required — the three core phenomena behind test-time compute scaling:
1) self-consistency gains are log-like with diminishing returns,
2) best-of-N *coverage* keeps paying even after majority vote plateaus,
3) majority vote has a modal ceiling and can *anti-scale* when the mode is wrong,


In [ ]:
import numpy as np
from scipy.stats import binom

def maj_vote_acc(N, p):
    """Exact expected majority-vote accuracy; ties broken at random."""
    need = N // 2 + 1
    acc = float(1.0 - binom.cdf(need - 1, N, p))
    if N % 2 == 0:
        acc += 0.5 * float(binom.pmf(N // 2, N, p))
    return acc

def coverage(N, p):
    """Exact P(at least one of N samples is correct)."""
    return float(1.0 - (1.0 - p) ** N)

## 1. Self-consistency: majority voting over N samples


In [ ]:
# bisection for the p that lands N=40 exactly on 0.744
lo, hi = 0.50, 0.62
for _ in range(60):
    mid = (lo + hi) / 2
    if maj_vote_acc(40, mid) < 0.744: lo = mid
    else: hi = mid
p = (lo + hi) / 2
print(f"calibrated p = {p:.4f}")
for N in [1, 2, 4, 8, 16, 32, 40]:
    print(f"N={N:3d}  majority-vote accuracy = {maj_vote_acc(N, p):.4f}")
# Note: the jump 1 -> 4 is the largest; 16 -> 40 barely moves. Diminishing returns.

## 2. Best-of-N coverage: the verifier's view
If a verifier can *recognize* one correct sample among N (Bay & Yearick, 2026), what matters


In [ ]:
for N in [1, 4, 16, 64, 256]:
    print(f"N={N:3d}  P(at least one correct) = {coverage(N, 0.3):.4f}")

## 3. The ceilings: majority vote can anti-scale, and samples are correlated
Two ceilings from Bay & Yearick (2026). First, the *modal* ceiling: on problems where the
model's most common answer is wrong (p=0.3), more samples sharpen the wrong answer —
accuracy *falls* as N grows. Second, the *correlation* ceiling: draws within one problem


In [ ]:
print('Modal ceiling (p=0.3: the mode is wrong):')
for N in [1, 8, 32, 128, 512]:
    print(f"N={N:3d}  majority-vote accuracy = {maj_vote_acc(N, 0.3):.4f}")
# Accuracy collapses: extra samples make the confident-wrong answer MORE certain.
print()
print('Correlation ceiling (rho=0.2):')
for n in [1, 10, 100, 1000]:
    n_eff = n / (1 + (n - 1) * 0.2)
    print(f"n={n:4d}  effective samples = {n_eff:.2f}")
# 1000 draws buy only ~5 effective samples. Past ~1/rho, samples are redundant.

## 4. The September 2026 twist: first is best
On hard benchmarks (AIME25, GPQA), the first solution in a trace is optimal 93.7% of the
time; later thinking fixes a wrong first answer only 2-7% of the time, but flips a right
answer wrong up to 21% of the time. Toy model below with the same qualitative dynamics
(first correct w.p. 0.65; each later solution flips right->wrong w.p. 0.15,


In [ ]:
rng = np.random.default_rng(42)
T = 300_000
first_ok = rng.random(T) < 0.65
rest = rng.random((T, 4))
rest_ok = np.where(first_ok[:, None], rest > 0.15, rest < 0.05)
sols = np.concatenate([first_ok[:, None], rest_ok], axis=1)
print(f"take FIRST     : {first_ok.mean():.4f}")
print(f"take LAST of 5 : {sols[:, -1].mean():.4f}")
print(f"MAJORITY of 5  : {(sols.sum(axis=1) >= 3).mean():.4f}")
# Takeaway: thinking longer mostly adds chances to talk yourself out of the right answer.
# The practical fix is a stopping rule: stop when confidence stops rising.

## References
- Snell et al. (2024) — scaling LLM test-time compute: 4x smaller model + optimal TTC matches larger baseline
- Wang et al. (2023) — self-consistency improves chain-of-thought reasoning
- Lightman et al. (2023) — PRM800K: step-level verification labels
- Bay & Yearick (2026) — sampling ceilings: modal and correlation ceilings of test-time scaling
